# Lab 2 — The Ingestion Pipeline & Schema Evolution

In Lab 1 you booted the stack and created a toy table. Now we do what data engineers actually do all day: **move data from an OLTP database into the lakehouse** — and survive the inevitable moment when the source team changes the schema without telling you.

In this lab you will:

1. **Extract** the `source_orders` table from PostgreSQL over JDBC.
2. **Load** it into an Iceberg table that uses **hidden partitioning** (`days(order_timestamp)`).
3. **Evolve the schema** — add a column, widen a type — with zero downtime and zero rewrites.
4. **Append a new-schema batch** and query old + new records side by side.

> **Prerequisite:** you must have seeded PostgreSQL first — see this lab's `README.md` (one `docker exec` command). The Lab 1 stack must be running.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Lab02-IngestionEvolution").getOrCreate()
print(f"Spark {spark.version} — default catalog: {spark.conf.get('spark.sql.defaultCatalog')}")

Spark 3.5.0 — default catalog: hive_prod


## Step 1 — Extract: read the source table over JDBC

The PostgreSQL JDBC driver is already on Spark's classpath (`org.postgresql:postgresql:42.6.0`, see `spark-defaults.conf`). Inside the Docker network the database is reachable at host **`postgres`**, port **5432**, database **`source_db`** — the same credentials as everything else in this course (`admin` / `password`).

This is the **E** in ELT: we pull the raw table as-is and let the lakehouse handle the rest.

In [3]:
df_orders = (
    spark.read.format("jdbc")
    .option("url", "jdbc:postgresql://postgres:5432/source_db")
    .option("dbtable", "source_orders")
    .option("user", "admin")
    .option("password", "password")
    .option("driver", "org.postgresql.Driver")
    .load()
)

print(f"Extracted {df_orders.count()} rows from PostgreSQL")
df_orders.printSchema()
df_orders.orderBy("order_id").show(5)

Extracted 15 rows from PostgreSQL
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_amount: decimal(10,2) (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)

+--------+-----------+------------+-------------------+
|order_id|customer_id|order_amount|    order_timestamp|
+--------+-----------+------------+-------------------+
|       1|        101|       49.99|2026-06-28 08:15:23|
|       2|        102|      129.50|2026-06-28 11:42:10|
|       3|        103|       15.00|2026-06-28 19:03:45|
|       4|        101|      220.75|2026-06-29 09:27:31|
|       5|        104|       75.20|2026-06-29 14:55:02|
+--------+-----------+------------+-------------------+
only showing top 5 rows



## Step 2 — Create the target table with *hidden partitioning*

In classic Hive tables, partitioning leaks into the schema: you'd add an artificial `order_date` column, writers must populate it, and any analyst who filters on `order_timestamp` instead of `order_date` accidentally triggers a **full table scan**.

Iceberg's **hidden partitioning** fixes this. We declare a *transform* — `days(order_timestamp)` — and Iceberg:
- derives the partition value from the column **automatically** on write,
- keeps the schema clean (no extra column!),
- derives partition predicates from ordinary filters on `order_timestamp` (equality, ranges, date comparisons) — no special partition column for analysts to know about.

The table keeps its logical schema; partitioning is pure metadata.

In [4]:
# Start from a clean slate so the lab can be re-run
spark.sql("DROP TABLE IF EXISTS hive_prod.db.orders_lakehouse PURGE")

spark.sql("""
    CREATE TABLE hive_prod.db.orders_lakehouse (
        order_id        INT,
        customer_id     INT,
        order_amount    DECIMAL(10,2),
        order_timestamp TIMESTAMP
    )
    USING iceberg
    PARTITIONED BY (days(order_timestamp))
""")

spark.sql("DESCRIBE TABLE hive_prod.db.orders_lakehouse").show(truncate=False)

+---------------+---------------------+-------+
|col_name       |data_type            |comment|
+---------------+---------------------+-------+
|order_id       |int                  |NULL   |
|customer_id    |int                  |NULL   |
|order_amount   |decimal(10,2)        |NULL   |
|order_timestamp|timestamp            |NULL   |
|               |                     |       |
|# Partitioning |                     |       |
|Part 0         |days(order_timestamp)|       |
+---------------+---------------------+-------+



Note the `# Partitioning` section above: `days(order_timestamp)` — and note there is **no** partition column in the schema itself.

## Step 3 — Load: write the DataFrame to Iceberg

We use the modern `DataFrameWriterV2` API (`.writeTo(...)`). `.append()` adds the rows as a new **snapshot** — remember from Lab 1: writes never modify existing files.

In [5]:
df_orders.writeTo("hive_prod.db.orders_lakehouse").append()

print(f"Rows in lakehouse table: {spark.table('hive_prod.db.orders_lakehouse').count()}")

Rows in lakehouse table: 15


Even though we never computed a partition value, Iceberg derived one per row. Check what it did:

In [6]:
spark.sql("""
    SELECT partition, record_count, file_count
    FROM hive_prod.db.orders_lakehouse.partitions
    ORDER BY partition
""").show(truncate=False)

+------------+------------+----------+
|partition   |record_count|file_count|
+------------+------------+----------+
|{2026-06-28}|3           |1         |
|{2026-06-29}|3           |1         |
|{2026-06-30}|3           |1         |
|{2026-07-01}|3           |1         |
|{2026-07-02}|3           |1         |
+------------+------------+----------+



One partition per calendar day, created invisibly. In MinIO you'll see matching folders like `data/order_timestamp_day=2026-06-28/`.

## Step 4 — The Twist: the source schema changes 😱

Two weeks into production, the commerce team ships a new feature and `source_orders` suddenly has a `discount_applied` flag. Amounts also outgrew `DECIMAL(10,2)`.

In a Hive/Parquet world this is a painful migration. In Iceberg, schema evolution is a **metadata-only** operation: every column has a permanent **field ID**, so adding/renaming/widening never rewrites data files and never breaks readers of old snapshots.

Evolve the table — both statements complete in milliseconds regardless of table size:

In [7]:
# 1. Add the new column (nullable by definition — old rows will read as NULL)
spark.sql("""
    ALTER TABLE hive_prod.db.orders_lakehouse
    ADD COLUMNS (discount_applied BOOLEAN COMMENT 'Was a discount applied at checkout?')
""")

# 2. Widen order_amount DECIMAL(10,2) -> DECIMAL(12,2): a safe, lossless type promotion
spark.sql("""
    ALTER TABLE hive_prod.db.orders_lakehouse
    ALTER COLUMN order_amount TYPE DECIMAL(12,2)
""")

# 3. Document the change for the next engineer
spark.sql("""
    ALTER TABLE hive_prod.db.orders_lakehouse
    ALTER COLUMN order_amount COMMENT 'Gross amount; widened to DECIMAL(12,2) in Lab 2'
""")

spark.sql("DESCRIBE TABLE hive_prod.db.orders_lakehouse").show(truncate=False)

+----------------+---------------------+-----------------------------------------------+
|col_name        |data_type            |comment                                        |
+----------------+---------------------+-----------------------------------------------+
|order_id        |int                  |NULL                                           |
|customer_id     |int                  |NULL                                           |
|order_amount    |decimal(12,2)        |Gross amount; widened to DECIMAL(12,2) in Lab 2|
|order_timestamp |timestamp            |NULL                                           |
|discount_applied|boolean              |Was a discount applied at checkout?            |
|                |                     |                                               |
|# Partitioning  |                     |                                               |
|Part 0          |days(order_timestamp)|                                               |
+----------------+---

> **Why is widening safe?** Every `DECIMAL(10,2)` value fits losslessly in `DECIMAL(12,2)`. Iceberg only allows *safe* promotions (`int→long`, `float→double`, decimal precision up). Narrowing is rejected — try it later if you're curious.

## Step 5 — Ingest a batch with the *new* schema

The next nightly batch arrives, now carrying the `discount_applied` flag (and bigger amounts). We build it as a mock DataFrame here — in production this would be the same JDBC read as Step 1.

In [8]:
from decimal import Decimal
from datetime import datetime
from pyspark.sql.types import (StructType, StructField, IntegerType,
                               DecimalType, TimestampType, BooleanType)

new_schema = StructType([
    StructField("order_id",         IntegerType(),    False),
    StructField("customer_id",      IntegerType(),    False),
    StructField("order_amount",     DecimalType(12,2), False),
    StructField("order_timestamp",  TimestampType(),  False),
    StructField("discount_applied", BooleanType(),    True),
])

new_batch = [
    (16, 111, Decimal("15499.00"), datetime(2026, 7, 3,  9, 14, 55), True),
    (17, 103, Decimal("42.75"),    datetime(2026, 7, 3, 12, 40, 18), False),
    (18, 112, Decimal("310.20"),   datetime(2026, 7, 3, 17, 5, 33),  True),
    (19, 105, Decimal("88.00"),    datetime(2026, 7, 4,  8, 22, 41), False),
    (20, 113, Decimal("1249.99"),  datetime(2026, 7, 4, 14, 58, 2),  True),
]

df_new_orders = spark.createDataFrame(new_batch, schema=new_schema)
df_new_orders.show()

# Append to the SAME table — no migration, no downtime
df_new_orders.writeTo("hive_prod.db.orders_lakehouse").append()

+--------+-----------+------------+-------------------+----------------+
|order_id|customer_id|order_amount|    order_timestamp|discount_applied|
+--------+-----------+------------+-------------------+----------------+
|      16|        111|    15499.00|2026-07-03 09:14:55|            true|
|      17|        103|       42.75|2026-07-03 12:40:18|           false|
|      18|        112|      310.20|2026-07-03 17:05:33|            true|
|      19|        105|       88.00|2026-07-04 08:22:41|           false|
|      20|        113|     1249.99|2026-07-04 14:58:02|            true|
+--------+-----------+------------+-------------------+----------------+



## Step 6 — Verification: old and new schemas, one query

The moment of truth. Old rows were written **before** `discount_applied` existed — Iceberg resolves them by field ID and returns `NULL` for the missing column. No backfill job, no `MSCK REPAIR`, no downtime.

In [9]:
spark.sql("""
    SELECT *
    FROM hive_prod.db.orders_lakehouse
    ORDER BY order_id
""").show(25)

spark.sql("""
    SELECT
        CASE WHEN discount_applied IS NULL THEN 'pre-evolution batch'
             ELSE 'post-evolution batch' END AS batch,
        COUNT(*)          AS rows,
        SUM(order_amount) AS total_amount
    FROM hive_prod.db.orders_lakehouse
    GROUP BY 1
""").show(truncate=False)

+--------+-----------+------------+-------------------+----------------+
|order_id|customer_id|order_amount|    order_timestamp|discount_applied|
+--------+-----------+------------+-------------------+----------------+
|       1|        101|       49.99|2026-06-28 08:15:23|            NULL|
|       2|        102|      129.50|2026-06-28 11:42:10|            NULL|
|       3|        103|       15.00|2026-06-28 19:03:45|            NULL|
|       4|        101|      220.75|2026-06-29 09:27:31|            NULL|
|       5|        104|       75.20|2026-06-29 14:55:02|            NULL|
|       6|        105|       33.10|2026-06-29 21:18:47|            NULL|
|       7|        102|      310.00|2026-06-30 07:44:12|            NULL|
|       8|        106|       58.60|2026-06-30 12:30:59|            NULL|
|       9|        103|       92.35|2026-06-30 16:22:08|            NULL|
|      10|        107|      480.99|2026-07-01 10:05:41|            NULL|
|      11|        104|       27.45|2026-07-01 13:37

## Bonus — the audit trail

Every step you just performed is recorded. Two appends → two snapshots; the schema change lives in the table metadata between them.

In [10]:
spark.sql("""
    SELECT committed_at, operation,
           summary['added-records']   AS added_records,
           summary['total-records']   AS total_records
    FROM hive_prod.db.orders_lakehouse.snapshots
    ORDER BY committed_at
""").show(truncate=False)

+-----------------------+---------+-------------+-------------+
|committed_at           |operation|added_records|total_records|
+-----------------------+---------+-------------+-------------+
|2026-07-10 06:50:56.089|append   |15           |15           |
|2026-07-10 06:50:59.538|append   |5            |20           |
+-----------------------+---------+-------------+-------------+



## ✅ Wrap-up

You have:
- ✔ Extracted an OLTP table into the lakehouse over JDBC (the E and L of ELT)
- ✔ Used **hidden partitioning** — day-level pruning with a clean schema
- ✔ Evolved a live table's schema (add column, widen type) as **metadata-only** operations
- ✔ Proved old and new data read together seamlessly, `NULL`-filling the new column

**Optional experiments before you leave:**
- Try to *narrow* the type back: `ALTER COLUMN order_amount TYPE DECIMAL(10,2)` → watch Iceberg refuse.
- Filter on the timestamp (`WHERE order_timestamp >= '2026-07-03'`) and check the Spark UI: how many files were scanned?

**Next (Lab 3):** Trino joins the stack — one copy of the data, many engines — plus **time travel** through the snapshots you've been accumulating.